# 03. Production Deployment Guide

This notebook covers deploying DiffML models to production environments.

## What you'll learn:
- Model optimization for inference
- Deployment strategies (REST API, batch, streaming)
- Performance monitoring
- Model versioning and A/B testing
- GPU optimization for production

## Prerequisites:
- Completed tutorials 01-02
- Basic understanding of web services
- Familiarity with Docker (helpful)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# For web service
from flask import Flask, request, jsonify
import requests

# Import DiffML
import sys
sys.path.append('..')
from src.diffml.models import DifferentialRegressor
from src.diffml.trainers import DifferentialTrainer
from src.diffml.datasets import BlackScholesDataset

print("✅ Libraries loaded")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 1. Model Optimization for Inference

Production models need to be fast and memory-efficient.

In [ ]:
# Train a model for deployment
print("Training model for deployment...")

# Generate data
dataset = BlackScholesDataset(n_samples=5000)
X, y, dy = dataset.generate()

# Create model
model = DifferentialRegressor(
    input_dim=5,
    hidden_units=[64, 64, 64],
    activation='relu'
)

# Quick training
trainer = DifferentialTrainer(model, differential_weight=0.5)
trainer.fit(X, y, dy, epochs=20, verbose=0)

print("✅ Model trained")
print(f"Model size: {sum(p.numel() for p in model.parameters()):,} parameters")

### 1.1 TorchScript Compilation

TorchScript provides optimized inference and language-agnostic deployment.

In [ ]:
# Convert to TorchScript
class ScriptableModel(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.model = original_model.model  # Get underlying nn.Module
        
    def forward(self, x):
        return self.model(x)

# Create scriptable version
scriptable = ScriptableModel(model)
scriptable.eval()

# Trace the model
example_input = torch.randn(1, 5)
traced_model = torch.jit.trace(scriptable, example_input)

# Save traced model
traced_model.save('model_traced.pt')

print("✅ Model traced and saved")

# Compare inference times
n_runs = 1000
test_input = torch.randn(100, 5)

# Original model
start = time.time()
for _ in range(n_runs):
    with torch.no_grad():
        _ = scriptable(test_input)
original_time = time.time() - start

# Traced model
start = time.time()
for _ in range(n_runs):
    with torch.no_grad():
        _ = traced_model(test_input)
traced_time = time.time() - start

print(f"\nOriginal model: {original_time:.3f}s")
print(f"Traced model: {traced_time:.3f}s")
print(f"Speedup: {original_time/traced_time:.2f}x")

### 1.2 Model Quantization

Reduce model size and improve inference speed with quantization.

In [ ]:
# Dynamic quantization
quantized_model = torch.quantization.quantize_dynamic(
    scriptable,
    {nn.Linear},  # Quantize Linear layers
    dtype=torch.qint8
)

# Compare model sizes
import os

torch.save(scriptable.state_dict(), 'model_original.pt')
torch.save(quantized_model.state_dict(), 'model_quantized.pt')

original_size = os.path.getsize('model_original.pt') / 1024
quantized_size = os.path.getsize('model_quantized.pt') / 1024

print(f"Original model size: {original_size:.2f} KB")
print(f"Quantized model size: {quantized_size:.2f} KB")
print(f"Size reduction: {(1 - quantized_size/original_size)*100:.1f}%")

# Test accuracy
test_X = torch.randn(100, 5)

with torch.no_grad():
    original_pred = scriptable(test_X)
    quantized_pred = quantized_model(test_X)

mse = torch.mean((original_pred - quantized_pred)**2)
print(f"\nQuantization MSE: {mse.item():.6f}")
print("✅ Quantization maintains accuracy!" if mse < 0.001 else "⚠️ Check quantization accuracy")

## 2. REST API Deployment

Deploy the model as a REST API for real-time pricing.

In [ ]:
# Create Flask application
api_code = '''
from flask import Flask, request, jsonify
import torch
import numpy as np

app = Flask(__name__)

# Load model at startup
model = torch.jit.load('model_traced.pt')
model.eval()

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy'})

@app.route('/price', methods=['POST'])
def price_option():
    try:
        # Parse request
        data = request.json
        
        # Extract parameters
        S = data.get('spot', 100.0)
        K = data.get('strike', 100.0)
        T = data.get('maturity', 1.0)
        r = data.get('rate', 0.05)
        sigma = data.get('volatility', 0.2)
        
        # Prepare input
        x = torch.tensor([[S, K, T, r, sigma]], dtype=torch.float32)
        
        # Inference
        with torch.no_grad():
            price = model(x).item()
        
        # Calculate Greeks (if needed)
        x.requires_grad_(True)
        price_grad = model(x)
        grads = torch.autograd.grad(price_grad, x)[0]
        
        return jsonify({
            'price': price,
            'delta': grads[0, 0].item(),
            'parameters': data
        })
        
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/batch_price', methods=['POST'])
def batch_price():
    try:
        data = request.json
        options = data.get('options', [])
        
        # Prepare batch
        batch = []
        for opt in options:
            batch.append([
                opt.get('spot', 100.0),
                opt.get('strike', 100.0),
                opt.get('maturity', 1.0),
                opt.get('rate', 0.05),
                opt.get('volatility', 0.2)
            ])
        
        x = torch.tensor(batch, dtype=torch.float32)
        
        # Batch inference
        with torch.no_grad():
            prices = model(x).numpy().tolist()
        
        return jsonify({
            'prices': prices,
            'count': len(prices)
        })
        
    except Exception as e:
        return jsonify({'error': str(e)}), 400

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

# Save API code
with open('api_server.py', 'w') as f:
    f.write(api_code)

print("✅ API server code created")
print("\nTo run the server:")
print("  python api_server.py")
print("\nAPI Endpoints:")
print("  GET  /health - Health check")
print("  POST /price - Single option pricing")
print("  POST /batch_price - Batch pricing")

### 2.1 API Client Example

In [ ]:
# Example API client
def call_pricing_api(spot, strike, maturity, rate, volatility, 
                     url='http://localhost:5000/price'):
    """
    Call the pricing API.
    """
    payload = {
        'spot': spot,
        'strike': strike,
        'maturity': maturity,
        'rate': rate,
        'volatility': volatility
    }
    
    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            return response.json()
        else:
            return {'error': response.text}
    except Exception as e:
        return {'error': str(e)}

# Test call (would work if server is running)
print("API Client Example:")
print("""result = call_pricing_api(
    spot=100.0,
    strike=105.0,
    maturity=0.25,
    rate=0.05,
    volatility=0.2
)""")

# Simulate response
mock_result = {
    'price': 2.3456,
    'delta': 0.4123,
    'parameters': {
        'spot': 100.0,
        'strike': 105.0,
        'maturity': 0.25,
        'rate': 0.05,
        'volatility': 0.2
    }
}

print(f"\nResponse: {json.dumps(mock_result, indent=2)}")

## 3. Docker Deployment

Package the model and API in a Docker container.

In [ ]:
# Create Dockerfile
dockerfile_content = '''
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model and code
COPY model_traced.pt .
COPY api_server.py .

# Expose port
EXPOSE 5000

# Health check
HEALTHCHECK --interval=30s --timeout=3s \
  CMD curl -f http://localhost:5000/health || exit 1

# Run server
CMD ["python", "api_server.py"]
'''

# Requirements file
requirements_content = '''
torch==2.0.0
flask==2.3.0
numpy==1.24.0
gunicorn==20.1.0
'''

# Docker compose for production
docker_compose = '''
version: '3.8'

services:
  diffml-api:
    build: .
    ports:
      - "5000:5000"
    environment:
      - PYTHONUNBUFFERED=1
      - OMP_NUM_THREADS=4
    restart: unless-stopped
    
  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf
    depends_on:
      - diffml-api
'''

print("📦 Docker configuration created")
print("\nTo build and run:")
print("  docker build -t diffml-api .")
print("  docker run -p 5000:5000 diffml-api")
print("\nOr with docker-compose:")
print("  docker-compose up")

## 4. Performance Monitoring

Monitor model performance in production.

In [ ]:
class ModelMonitor:
    """
    Monitor model performance and detect drift.
    """
    def __init__(self, model, baseline_data):
        self.model = model
        self.baseline_stats = self._compute_stats(baseline_data)
        self.metrics_history = []
        
    def _compute_stats(self, data):
        """Compute baseline statistics."""
        return {
            'mean': data.mean(dim=0),
            'std': data.std(dim=0),
            'min': data.min(dim=0)[0],
            'max': data.max(dim=0)[0]
        }
    
    def check_drift(self, new_data, threshold=3.0):
        """Check for distribution drift."""
        new_stats = self._compute_stats(new_data)
        
        # Calculate z-scores
        z_scores = torch.abs(
            (new_stats['mean'] - self.baseline_stats['mean']) / 
            (self.baseline_stats['std'] + 1e-8)
        )
        
        drift_detected = (z_scores > threshold).any()
        
        return {
            'drift_detected': drift_detected.item(),
            'z_scores': z_scores.tolist(),
            'max_z_score': z_scores.max().item()
        }
    
    def log_prediction(self, input_data, prediction, latency):
        """Log prediction metrics."""
        self.metrics_history.append({
            'timestamp': time.time(),
            'input_mean': input_data.mean().item(),
            'prediction': prediction.item() if prediction.numel() == 1 else prediction.mean().item(),
            'latency_ms': latency * 1000
        })
    
    def get_metrics_summary(self):
        """Get summary of recent metrics."""
        if not self.metrics_history:
            return {}
        
        recent = self.metrics_history[-100:]  # Last 100 predictions
        latencies = [m['latency_ms'] for m in recent]
        
        return {
            'avg_latency_ms': np.mean(latencies),
            'p95_latency_ms': np.percentile(latencies, 95),
            'p99_latency_ms': np.percentile(latencies, 99),
            'total_predictions': len(self.metrics_history)
        }

# Create monitor
baseline_data = torch.randn(1000, 5)  # Historical data
monitor = ModelMonitor(model, baseline_data)

# Simulate production usage
print("Simulating production monitoring...\n")

for i in range(100):
    # Generate request (with potential drift)
    if i > 50:
        # Introduce drift
        test_input = torch.randn(1, 5) * 1.5 + 0.5
    else:
        test_input = torch.randn(1, 5)
    
    # Time prediction
    start = time.time()
    with torch.no_grad():
        pred = model.model(test_input)
    latency = time.time() - start
    
    # Log metrics
    monitor.log_prediction(test_input, pred, latency)
    
    # Check drift periodically
    if i % 20 == 0 and i > 0:
        recent_inputs = torch.randn(20, 5) if i < 50 else torch.randn(20, 5) * 1.5 + 0.5
        drift_result = monitor.check_drift(recent_inputs)
        
        if drift_result['drift_detected']:
            print(f"⚠️ Step {i}: Drift detected! Max z-score: {drift_result['max_z_score']:.2f}")

# Get metrics summary
summary = monitor.get_metrics_summary()
print("\n📊 Performance Summary:")
for key, value in summary.items():
    print(f"  {key}: {value:.3f}" if isinstance(value, float) else f"  {key}: {value}")

## 5. Model Versioning and A/B Testing

In [ ]:
class ModelRegistry:
    """
    Manage multiple model versions for A/B testing.
    """
    def __init__(self):
        self.models = {}
        self.metrics = {}
        self.traffic_split = {}
        
    def register_model(self, name, model, version, metadata=None):
        """Register a new model version."""
        key = f"{name}_v{version}"
        self.models[key] = {
            'model': model,
            'version': version,
            'metadata': metadata or {},
            'created': time.time()
        }
        self.metrics[key] = {'requests': 0, 'errors': 0, 'total_latency': 0}
        print(f"✅ Registered model: {key}")
        
    def set_traffic_split(self, splits):
        """Set traffic distribution for A/B testing."""
        total = sum(splits.values())
        self.traffic_split = {k: v/total for k, v in splits.items()}
        print(f"Traffic split updated: {self.traffic_split}")
        
    def get_model(self, strategy='random'):
        """Get model based on strategy."""
        if strategy == 'random' and self.traffic_split:
            # A/B testing
            r = np.random.random()
            cumsum = 0
            for model_key, prob in self.traffic_split.items():
                cumsum += prob
                if r < cumsum:
                    return model_key, self.models[model_key]['model']
        
        # Default to latest
        latest_key = max(self.models.keys(), 
                        key=lambda k: self.models[k]['version'])
        return latest_key, self.models[latest_key]['model']
    
    def log_request(self, model_key, latency, error=False):
        """Log request metrics."""
        self.metrics[model_key]['requests'] += 1
        self.metrics[model_key]['total_latency'] += latency
        if error:
            self.metrics[model_key]['errors'] += 1
    
    def get_comparison_metrics(self):
        """Compare model performance."""
        comparison = []
        for key, metrics in self.metrics.items():
            if metrics['requests'] > 0:
                comparison.append({
                    'model': key,
                    'requests': metrics['requests'],
                    'avg_latency_ms': metrics['total_latency'] / metrics['requests'] * 1000,
                    'error_rate': metrics['errors'] / metrics['requests']
                })
        return pd.DataFrame(comparison)

# Create registry
registry = ModelRegistry()

# Register multiple model versions
# Version 1: Original model
registry.register_model('diffml', model, version=1, 
                       metadata={'description': 'Original model'})

# Version 2: Smaller model
model_v2 = DifferentialRegressor(
    input_dim=5,
    hidden_units=[32, 32],  # Smaller
    activation='relu'
)
registry.register_model('diffml', model_v2, version=2,
                       metadata={'description': 'Lightweight model'})

# Set up A/B test: 70% v1, 30% v2
registry.set_traffic_split({
    'diffml_v1': 0.7,
    'diffml_v2': 0.3
})

# Simulate A/B testing
print("\n🧪 Running A/B test simulation...\n")

for i in range(200):
    # Get model based on traffic split
    model_key, selected_model = registry.get_model(strategy='random')
    
    # Generate request
    test_input = torch.randn(10, 5)  # Batch request
    
    # Time inference
    start = time.time()
    try:
        with torch.no_grad():
            pred = selected_model.model(test_input) if hasattr(selected_model, 'model') else selected_model(test_input)
        latency = time.time() - start
        error = False
    except Exception as e:
        latency = time.time() - start
        error = True
    
    # Log metrics
    registry.log_request(model_key, latency, error)

# Show comparison
comparison_df = registry.get_comparison_metrics()
print("\n📊 A/B Test Results:")
display(comparison_df)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Request distribution
ax1.bar(comparison_df['model'], comparison_df['requests'])
ax1.set_xlabel('Model Version')
ax1.set_ylabel('Number of Requests')
ax1.set_title('Request Distribution')

# Latency comparison
ax2.bar(comparison_df['model'], comparison_df['avg_latency_ms'])
ax2.set_xlabel('Model Version')
ax2.set_ylabel('Average Latency (ms)')
ax2.set_title('Latency Comparison')

plt.tight_layout()
plt.show()

## 6. GPU Optimization for Production

In [ ]:
# GPU optimization utilities
class GPUOptimizer:
    """
    Optimize model for GPU inference.
    """
    @staticmethod
    def optimize_batch_size(model, input_shape, device='cuda'):
        """Find optimal batch size for GPU."""
        if not torch.cuda.is_available():
            return 256  # Default for CPU
        
        model = model.to(device)
        model.eval()
        
        batch_sizes = [1, 8, 16, 32, 64, 128, 256, 512, 1024]
        throughputs = []
        
        for batch_size in batch_sizes:
            try:
                x = torch.randn(batch_size, *input_shape[1:]).to(device)
                
                # Warm up
                for _ in range(10):
                    with torch.no_grad():
                        _ = model(x)
                
                # Measure
                torch.cuda.synchronize()
                start = time.time()
                
                for _ in range(100):
                    with torch.no_grad():
                        _ = model(x)
                
                torch.cuda.synchronize()
                elapsed = time.time() - start
                
                throughput = (batch_size * 100) / elapsed
                throughputs.append(throughput)
                
            except RuntimeError:  # Out of memory
                break
        
        optimal_idx = np.argmax(throughputs)
        return batch_sizes[optimal_idx], throughputs
    
    @staticmethod
    def enable_mixed_precision(model):
        """Enable automatic mixed precision."""
        from torch.cuda.amp import autocast
        
        class AMPWrapper(nn.Module):
            def __init__(self, model):
                super().__init__()
                self.model = model
            
            @autocast()
            def forward(self, x):
                return self.model(x)
        
        return AMPWrapper(model)

# Test GPU optimization
if torch.cuda.is_available():
    print("🎯 GPU Optimization Analysis\n")
    
    optimizer = GPUOptimizer()
    
    # Find optimal batch size
    optimal_batch, throughputs = optimizer.optimize_batch_size(
        model.model, (None, 5)
    )
    
    print(f"Optimal batch size: {optimal_batch}")
    
    # Plot throughput curve
    batch_sizes = [1, 8, 16, 32, 64, 128, 256, 512, 1024][:len(throughputs)]
    
    plt.figure(figsize=(10, 6))
    plt.plot(batch_sizes, throughputs, 'o-', linewidth=2, markersize=8)
    plt.xlabel('Batch Size')
    plt.ylabel('Throughput (samples/sec)')
    plt.title('GPU Throughput vs Batch Size')
    plt.xscale('log')
    plt.grid(True, alpha=0.3)
    plt.axvline(x=optimal_batch, color='r', linestyle='--', label=f'Optimal: {optimal_batch}')
    plt.legend()
    plt.show()
else:
    print("⚠️ GPU not available. Using CPU optimization strategies.")
    print("\nCPU Optimization Tips:")
    print("  1. Use smaller batch sizes (32-256)")
    print("  2. Enable multi-threading with OMP_NUM_THREADS")
    print("  3. Consider model quantization")
    print("  4. Use TorchScript for faster inference")

## 7. Production Checklist

Before deploying to production, ensure:

In [ ]:
# Production readiness checklist
checklist = [
    ("Model Performance", [
        "✅ Model accuracy meets requirements",
        "✅ Latency < target SLA",
        "✅ Throughput tested under load",
        "✅ Memory usage acceptable"
    ]),
    ("Model Optimization", [
        "✅ TorchScript compilation",
        "⚠️ Quantization (if needed)",
        "✅ Batch size optimized",
        "⚠️ Mixed precision (for GPU)"
    ]),
    ("Deployment", [
        "✅ Docker container built",
        "✅ Health checks implemented",
        "✅ Logging configured",
        "⚠️ Load balancing setup"
    ]),
    ("Monitoring", [
        "✅ Metrics collection",
        "✅ Drift detection",
        "⚠️ Alerting rules",
        "⚠️ Dashboard created"
    ]),
    ("Testing", [
        "✅ Unit tests pass",
        "✅ Integration tests pass",
        "⚠️ Load testing complete",
        "⚠️ Failure scenarios tested"
    ]),
    ("Security", [
        "⚠️ API authentication",
        "⚠️ Input validation",
        "⚠️ Rate limiting",
        "⚠️ SSL/TLS configured"
    ])
]

print("📋 PRODUCTION READINESS CHECKLIST\n")
print("=" * 50)

for category, items in checklist:
    print(f"\n{category}:")
    for item in items:
        print(f"  {item}")

print("\n" + "=" * 50)
print("\n✅ = Complete  ⚠️ = Needs attention")

## 8. Cost Optimization

Optimize deployment costs while maintaining performance.

In [ ]:
# Cost analysis
def calculate_deployment_cost(requests_per_month, avg_latency_ms, 
                             instance_type='cpu'):
    """
    Estimate deployment costs for different configurations.
    """
    # Pricing (example rates)
    pricing = {
        'cpu': {'hourly': 0.0464, 'name': 't3.medium'},
        'gpu': {'hourly': 0.526, 'name': 'g4dn.xlarge'},
        'serverless': {'per_million': 20.0, 'per_gb_sec': 0.0000166667}
    }
    
    results = {}
    
    # Instance-based deployment
    for itype in ['cpu', 'gpu']:
        # Calculate required instances
        requests_per_sec = requests_per_month / (30 * 24 * 3600)
        throughput_per_instance = 1000 / avg_latency_ms  # requests/sec
        
        instances_needed = max(1, int(np.ceil(requests_per_sec / throughput_per_instance)))
        
        monthly_cost = instances_needed * pricing[itype]['hourly'] * 24 * 30
        
        results[itype] = {
            'instance_type': pricing[itype]['name'],
            'instances': instances_needed,
            'monthly_cost': monthly_cost,
            'cost_per_1k_requests': (monthly_cost / requests_per_month) * 1000
        }
    
    # Serverless deployment
    memory_gb = 0.5  # Assumed memory usage
    compute_time_sec = avg_latency_ms / 1000
    
    request_cost = (requests_per_month / 1e6) * pricing['serverless']['per_million']
    compute_cost = requests_per_month * compute_time_sec * memory_gb * pricing['serverless']['per_gb_sec']
    
    results['serverless'] = {
        'monthly_cost': request_cost + compute_cost,
        'cost_per_1k_requests': ((request_cost + compute_cost) / requests_per_month) * 1000
    }
    
    return results

# Calculate costs for different scenarios
scenarios = [
    ("Low Traffic", 100_000, 10),
    ("Medium Traffic", 1_000_000, 10),
    ("High Traffic", 10_000_000, 10)
]

print("💰 DEPLOYMENT COST ANALYSIS\n")
print("=" * 70)

for scenario_name, requests, latency in scenarios:
    print(f"\n{scenario_name}: {requests:,} requests/month")
    print("-" * 40)
    
    costs = calculate_deployment_cost(requests, latency)
    
    # Create comparison table
    cost_df = pd.DataFrame([
        {
            'Deployment': 'CPU Instance',
            'Type': costs['cpu']['instance_type'],
            'Count': costs['cpu']['instances'],
            'Monthly Cost': f"${costs['cpu']['monthly_cost']:.2f}",
            'Per 1K Requests': f"${costs['cpu']['cost_per_1k_requests']:.3f}"
        },
        {
            'Deployment': 'GPU Instance',
            'Type': costs['gpu']['instance_type'],
            'Count': costs['gpu']['instances'],
            'Monthly Cost': f"${costs['gpu']['monthly_cost']:.2f}",
            'Per 1K Requests': f"${costs['gpu']['cost_per_1k_requests']:.3f}"
        },
        {
            'Deployment': 'Serverless',
            'Type': 'Lambda/Functions',
            'Count': '-',
            'Monthly Cost': f"${costs['serverless']['monthly_cost']:.2f}",
            'Per 1K Requests': f"${costs['serverless']['cost_per_1k_requests']:.3f}"
        }
    ])
    
    display(cost_df)

print("\n📊 Recommendations:")
print("  • Low traffic: Consider serverless for cost efficiency")
print("  • Medium traffic: CPU instances with auto-scaling")
print("  • High traffic: Dedicated instances with load balancing")
print("  • Use spot instances for 70% cost reduction (with fallback)")

## Summary

You've learned how to deploy DiffML models to production:

✅ **Model Optimization**: TorchScript, quantization, batching

✅ **Deployment Options**: REST API, Docker, serverless

✅ **Monitoring**: Performance tracking, drift detection

✅ **A/B Testing**: Version management, traffic splitting

✅ **Cost Optimization**: Instance selection, auto-scaling

### Best Practices:

1. **Start Small**: Deploy to staging first
2. **Monitor Everything**: Latency, accuracy, drift
3. **Plan for Failure**: Implement fallbacks and circuit breakers
4. **Optimize Iteratively**: Profile → Optimize → Measure
5. **Document Well**: API specs, runbooks, architecture

### Next Steps:

- Set up CI/CD pipeline
- Implement comprehensive monitoring
- Create load testing scenarios
- Build model retraining pipeline
- Establish SLAs and alerting

---

*Ready for production? Your DiffML models are now enterprise-ready!*